# ECMWF TC Storm Surge Pipeline

This notebook demonstrates the storm surge step (step 5c): from the ECMWF ensemble tracks and 10 m wind
to one GeoTIFF per storm and forecast cycle with the maximum surge depth of each ensemble member.

## Overview

| Step | Module | Output |
|---|---|---|
| 0. Coastal elevation cache (one-time) | `setup_surge_dem` | `surge_coastal_dem.tif`: DeltaDTM elevation + distance to coast of low-lying coastal land |
| 2–3. Tracks | `ecmwf_tc_data_extractor`, `ecmwf_tc_data_transformer` | Transformed track CSV |
| 4. Wind | `ecmwf_wind_data_downloader` (here: a local GRIB split into the same files) | `wind_ens_*_pf/cf.grib2` |
| 5c. Surge raster | `ecmwf_surge_envelope_extractor` | `<STORM>_<YYYYMMDDTHHZ>_surge_raster.tif`, uploaded to `surge/` |

## Method

Surge is computed with the CLIMADA `TCSurgeBathtub` method, re-implemented with numpy
(equivalence checked in `tests/test_surge_bathtub.py`):

* surge = 0.1023 × max(V − 26.8224, 0) + 1.8288 − elevation − 0.2 × distance to coast (km), with V the 10 m wind in m/s (Xu 2010)
* computed only on coastal land between 0 and 10 m and within 50 km of the coast, where V ≥ 34 kt
* for each of the 51 members and each lead time, on a 15″ (~450 m) grid, then the maximum over lead times

The raster has one band per member (bands 1–50: perturbed members, band 51: control), with the maximum surge depth
in m: 0 on a coastal cell that is not flooded, nodata everywhere else. No threshold is applied: the exposure step
chooses its own.

## Test case

Hurricane Melissa, Jamaica, ECMWF forecast of 28 October 2025, 00 UTC, lead times 0 to 48 h.
The run uses files already on disk (BUFR tracks, ENS 10 m wind GRIB, DeltaDTM for Jamaica), since ECMWF open data
only keeps the last few days online. For a live cycle, the same step runs inside `github_actions/main.py`
with `PROCESS_SURGE=true`.

## Setup

In [ ]:
import os, sys, time, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

from ecmwf_tc_data_extractor import extract_tc_data, filter_tc_data, save_per_storm_csvs
from ecmwf_tc_data_transformer import transform_tc_data_from_file
from ecmwf_surge_envelope_extractor import process_surge_combination, SURGE_DEM_FILENAME

timings = {}

In [ ]:
# ── Inputs already on disk (sensitivity test data) ──────────────────────────────
DATA = Path("../stormsurge/data")
BUFR_FILE = DATA / "bufr/20251028_00z_tf.bufr"
WIND_GRIB = DATA / "wind/ecmwf_enfo_10uv_20251028_00z_0-48h.grib2"
DEM_FILE  = DATA / "dem/JAM_deltadtm.tif"        # DeltaDTM for Jamaica (tests/surge_model_comparison/notebooks/00_coastal_dem_gee.ipynb)

STORM = "MELISSA"
RUN_DATE, RUN_HOUR = "2025-10-28", 0
BBOX = (-78.5, 17.6, -76.1, 18.6)                # Jamaica (west, south, east, north)

# ── Working directories (same layout as the pipeline) ─────────────────────
WORK = Path("demo_surge")
directories = {k: WORK / k for k in ("tc_data", "tc_transformed", "wind_data", "wind_extracted", "surge_dem")}
for d in directories.values():
    d.mkdir(parents=True, exist_ok=True)

for f in (BUFR_FILE, WIND_GRIB, DEM_FILE):
    print(f"{'OK     ' if f.exists() else 'MISSING'} {f}")

## Step 0: Coastal elevation cache (one-time)

In production, `setup_surge_dem.py` is run once for the whole 56S–60N band: it downloads DeltaDTM from
4TU.ResearchData and stores the cache on the Snowflake stage or on Blob Storage, under `surge/coastal_dem/v1/`.
Here it is built for Jamaica only, from the DeltaDTM file already on disk (`--dem`), so nothing is downloaded.

In [ ]:
t0 = time.time()
cmd = [sys.executable, "setup_surge_dem.py",
       "--bbox", *map(str, BBOX),
       "--dem", str(DEM_FILE),
       "--dest-dir", str(WORK / "surge_dem_tmp"),
       "--local-only", str(directories["surge_dem"])]
print(" ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stderr[-2500:])
res.check_returncode()
timings["0_dem_cache"] = time.time() - t0

DEM_CACHE = directories["surge_dem"] / SURGE_DEM_FILENAME
with rasterio.open(DEM_CACHE) as src:
    elev = src.read(1)
    print(f"Cache grid: {src.width} x {src.height}, {src.res[0] * 3600:.0f}\" | "
          f"coastal lowland cells: {np.isfinite(elev).sum():,}")

## Steps 2–3: Tracks (pipeline extractor and transformer)

In [ ]:
t0 = time.time()
df = filter_tc_data(extract_tc_data(str(BUFR_FILE), verbose=False), named_storms_only=True)
print("Named storms:", list(df["storm_id"].unique()))
csvs = [p for p in save_per_storm_csvs(df, str(directories["tc_data"]), BUFR_FILE.name, verbose=False)
        if f"_storm_{STORM}_" in Path(p).name]
for p in csvs:
    r = transform_tc_data_from_file(filename=p, output_dir=str(directories["tc_transformed"]), verbose=False)
    print(f"{r['records']} track records -> {Path(r['csv_file']).name}")
timings["2_3_tracks"] = time.time() - t0

## Step 4: Wind

In production, step 4 downloads one GRIB per lead time and per type (`wind_ens_<date>_r<HH>_f<lead>h_pf.grib2`
for the 50 perturbed members, `_cf` for the control). Here the local 0–48 h GRIB is split into exactly those files.

In [ ]:
import eccodes

t0 = time.time()
handles = {}
with open(WIND_GRIB, "rb") as f:
    while (gid := eccodes.codes_grib_new_from_file(f)) is not None:
        step = int(eccodes.codes_get(gid, "step"))
        kind = eccodes.codes_get(gid, "dataType")           # "cf" or "pf"
        name = f"wind_ens_{RUN_DATE}_r{RUN_HOUR:02d}_f{step:03d}h_{kind}.grib2"
        if name not in handles:
            handles[name] = open(directories["wind_data"] / name, "wb")
        eccodes.codes_write(gid, handles[name])
        eccodes.codes_release(gid)
for h in handles.values():
    h.close()
timings["4_wind_split"] = time.time() - t0
print(f"{len(handles)} wind files written in {timings['4_wind_split']:.0f} s")

## Step 5c: Surge raster

In [ ]:
t0 = time.time()
result = process_surge_combination(
    tc_data_dir=directories["tc_transformed"],
    wind_data_dir=directories["wind_data"],
    output_dir=directories["wind_extracted"],
    dem_path=DEM_CACHE,
)
timings["5c_surge"] = time.time() - t0
RASTER = result["raster_files"][0]
print(f"{RASTER.name} | {result['timings']}")

In [ ]:
with rasterio.open(RASTER) as src:
    depth = src.read()                                   # (51 members, rows, cols), m; NaN = not coastal
    r_transform, r_extent = src.transform, (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
    tags, res_arcsec = src.tags(), src.res[0] * 3600
coastal = np.isfinite(depth[0])
print(f"{depth.shape[0]} bands | grid {depth.shape[2]} x {depth.shape[1]}, {res_arcsec:.0f}\" | "
      f"{coastal.sum():,} coastal cells | lead times {tags['lead_times_h']} h | elevation: {tags['elevation_source']}")

# Members reaching each depth somewhere on the coast (the thresholds are only for this summary)
summary = pd.DataFrame({
    "members reaching depth": [int((np.nanmax(depth, axis=(1, 2)) >= thr).sum()) for thr in (0.5, 1, 2, 3)],
}, index=["0.5 m", "1 m", "2 m", "3 m"])
summary["probability"] = (summary.iloc[:, 0] / depth.shape[0]).round(2)
summary["control max depth (m)"] = round(float(np.nanmax(depth[-1])), 2)
summary

In [ ]:
# Maximum depth per member (band 51 = control)
per_member = pd.Series(np.nanmax(depth, axis=(1, 2)).round(2), index=[f"member_{m}" for m in tags["member_numbers"].split(",")],
                       name="max depth (m)")
per_member.describe().round(2)

## Results: probability of surge per cell

What the exposure step does with the raster: for each cell, the probability is the share of the 51 members whose
depth reaches the threshold.

In [ ]:
tracks = pd.read_csv(next(directories["tc_transformed"].glob("*_transformed.csv")))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for ax, thr in zip(axes, (0.5, 1.0)):
    prob = np.where(coastal, (depth >= thr).mean(axis=0), np.nan)
    prob[prob == 0] = np.nan
    ax.imshow(np.where(coastal, 0.0, np.nan), extent=r_extent, cmap="Greys", vmin=0, vmax=3)
    im = ax.imshow(prob, extent=r_extent, cmap="Blues", vmin=0, vmax=1)
    for _, t in tracks[tracks["lead_time"] <= 48].sort_values("lead_time").groupby("ensemble_member"):
        ax.plot(t["longitude"], t["latitude"], color="crimson", lw=0.4, alpha=0.4)
    ax.set_xlim(BBOX[0], BBOX[2]); ax.set_ylim(BBOX[1], BBOX[3])
    ax.set_title(f"P(surge ≥ {thr:g} m)")
fig.colorbar(im, ax=axes, shrink=0.8, label="share of the 51 members")
fig.suptitle(f"{STORM}, ECMWF forecast {RUN_DATE} {RUN_HOUR:02d} UTC, 0–48 h. Grey: coastal land below 10 m", y=0.98)
plt.savefig(WORK / "surge_probability_map.png", dpi=150, bbox_inches="tight")
plt.show()

## Runtime

In [ ]:
pd.Series(timings, name="seconds").round(1).to_frame()

In production only step 5c runs on each cycle: the elevation cache is built once, and the tracks and wind
are already produced by steps 2–4 for the wind and gust envelopes.